Relier la borne de tension de sortie Vout du capteur de pression à la borne de mesure EA0 de la carte Arduino.

Le programme à envoyer à Arduino est acquisition_continue.ino

On commence par importer les bibliothèques nécessaires. Attention : NE PAS EFFACER !

In [ ]:
%run Import_bibliotheques.ipynb

%run Arduino.ipynb

connexion_arduino()

In [ ]:
nom = "variable"    # Donner entre les guillemets un nom pertinent à la variable dont dépend la pression

Acquisition des valeurs.  
Entrer une nouvelle valeur de la variable pour réaliser une nouvelle mesure. Pour arrêter, valider sans entrer de valeur.

In [ ]:
valeurs = [nom+",P"]
# L'étalonnage du capteur est du type P = coef * Vout + décalage ; les valeurs suivantes sont celles données par le constructeur pour le capteur de pression MPX4250AP
coef = 5e4
decalage = 1e4
while True :
    entree = input("Entrer la valeur du paramètre (entrer une ligne vide pour arrêter) : ")
    if entree.strip() == "" :
        break
    try :
        valeur = float(entree.replace(',', '.'))
    except ValueError :
        print("Valeur invalide : veuillez recommencer.")
        continue
    try :
        if 'ser' not in globals() or not ser.is_open :
            connexion_arduino()
        # L'attente de saisie laisse s'accumuler des données ; mesure_unique() les vide
        tension = mesure_unique()
    except Exception as erreur :
        print(f"Erreur avec l'Arduino : {erreur}")
        break
    pression = coef * tension + decalage
    print(f"pression mesurée : {pression:.2f} Pa")
    valeurs.append(f"{valeur},{pression}")
    with open("pression.csv", 'w') as fichier :
        fichier.write("\n".join(valeurs) + "\n")

Importation de tableaux enregistrés localement en fichiers csv :

In [ ]:
tableurVersVariables('pression.csv')

Tracé de graphique(s) à partir des tableaux précédents :

In [ ]:
abscisse = nom    # choix de la grandeur en abscisse, pour le graphique et pour le modèle
ordonnee = "P"    # choix de la grandeur en ordonnée, pour le graphique et pour le modèle
equation = "a * x + b"    # expression de l'équation correspondant au modèle recherché ; la variable peut être x ou le nom donné en abscisse
debut = None    # indice du premier élément à prendre en compte pour le modèle (None : le premier de la liste)
fin = None    # indice du dernier élément à prendre en compte pour le modèle (None : le dernier de la liste)
debutCourbe = None    # valeur de l'abscisse où débute le tracé du modèle (None : la plus petite valeur de l'abscisse)
finCourbe = None    # valeur de l'abscisse où se termine le tracé du modèle (None : la plus grande valeur de l'abscisse)
contraintes = [['a', 1, True, -inf, inf]]    # ['nomDuParametre', valeurInitiale, variabilité, min, max]   à remplir pour chaque paramètre du modèle si la recherche automatique ne trouve pas de valeurs cohérentes

xMod, yMod, expression, valeurs, abscisse, ordonnee, modele, parametres = Calcul_modele(abscisse, ordonnee, equation, debut, fin, debutCourbe, finCourbe, contraintes)    # Calcul des valeurs du modèle, ne pas modifier

# Tracé des points expérimentaux :
fig = go.Figure()
fig.add_trace(go.Scatter(x=abscisse, y=ordonnee, mode="markers", name="",
    marker=dict(symbol="cross", size=10, color="blue")))

# Tracé du modèle :
fig.add_trace(go.Scatter(x=xMod, y=yMod, mode="lines", name=f"Modèle : {expression}<br>{valeurs.strip().replace(chr(10), '<br>')}",
    line=dict(color="crimson", width=2)))

# Cosmétique du graphique :
#fig.update_yaxes(scaleanchor="x", scaleratio=1)    # à décommenter pour un repère orthonormé
fig.update_xaxes(showgrid=True)    # grille verticale (par défaut)
fig.update_yaxes(showgrid=True)    # grille horizontale (par défaut)
fig.update_layout(
    showlegend=True,    # légende du graphique,
    title="Mon titre",    # mettre le titre désiré entre les guillemets,
    xaxis_title="",    # nom de l'axe des abscisses,
    yaxis_title="",    # nom de l'axe des ordonnées,
    height=600)    # hauteur du graphique en pixels ; ex. : 450, 700...

Calcul de valeurs particulières :